In [9]:
import json
from pathlib import Path

In [10]:
NPL_DIR = Path("npl_json")
files = sorted(NPL_DIR.glob("*.json"))
print(NPL_DIR)
print(files[0])


npl_json
npl_json\1462596.json


In [11]:
with open(files[0],encoding="utf-8") as f:
    match = json.load(f)

print(match.keys())
print(match["info"]["teams"])
print(match["innings"][0].keys())

dict_keys(['meta', 'info', 'innings'])
['Biratnagar Kings', 'Janakpur Bolts']
dict_keys(['team', 'overs', 'powerplays'])


In [12]:
first_over = match["innings"][0]["overs"][0]
print(first_over["over"])
for delivery in first_over["deliveries"]:
    print(delivery)

0
{'actual_delivery': '0.1', 'batter': 'Lokesh Bam', 'bowler': 'K Mahato', 'non_striker': 'MJ Guptill', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}
{'actual_delivery': '0.2', 'batter': 'Lokesh Bam', 'bowler': 'K Mahato', 'non_striker': 'MJ Guptill', 'runs': {'batter': 4, 'extras': 0, 'total': 4}}
{'actual_delivery': '0.3', 'batter': 'Lokesh Bam', 'bowler': 'K Mahato', 'non_striker': 'MJ Guptill', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}
{'actual_delivery': '0.4', 'batter': 'Lokesh Bam', 'bowler': 'K Mahato', 'extras': {'wides': 1}, 'non_striker': 'MJ Guptill', 'runs': {'batter': 0, 'extras': 1, 'total': 1}}
{'actual_delivery': '0.4', 'batter': 'Lokesh Bam', 'bowler': 'K Mahato', 'non_striker': 'MJ Guptill', 'runs': {'batter': 0, 'extras': 0, 'total': 0}}
{'actual_delivery': '0.5', 'batter': 'Lokesh Bam', 'bowler': 'K Mahato', 'non_striker': 'MJ Guptill', 'runs': {'batter': 1, 'extras': 0, 'total': 1}}
{'actual_delivery': '0.6', 'batter': 'MJ Guptill', 'bowler': 'K Mahato', '

In [13]:
def load_match(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

match = load_match(files[0])
print(match["info"]["teams"]) 

['Biratnagar Kings', 'Janakpur Bolts']


In [14]:
def is_legal(delivery):
    extras = delivery.get("extras",{})
    return "wides" not in extras and "noballs" not in extras

In [15]:
print(is_legal({"runs": {"total": 1}}))         # normal ball
print(is_legal({"extras": {"wides": 1}}))       # wide
print(is_legal({"extras": {"noballs": 1}}))     # no-ball
print(is_legal({"extras": {"legbyes": 2}}))     # leg byes


True
False
False
True


In [16]:
def count_legal_balls(innings):
    count = 0
    for over in innings["overs"]:
        for delivery in over["deliveries"]:
            if is_legal(delivery):
                count +=1
    return count

print(count_legal_balls(match["innings"][0]))
print(count_legal_balls(match["innings"][1]))

114
93


In [17]:
problems = []
innings_seen = 0

for path in files:
    match = load_match(path)
    for i, innings in enumerate(match["innings"]):
        innings_seen += 1
        balls = count_legal_balls(innings)
        if balls > 120:
            problems.append((path.stem, i + 1, balls))

print("matches:", len(files))
print("innings:", innings_seen)
print("problems:", problems)


matches: 64
innings: 130
problems: []


In [24]:
problems = []
innings_seen = 0

for path in files:
    match = load_match(path)
    if len(match["innings"]) != 2:
        print(path.stem, len(match["innings"]), match["info"]["outcome"])



1462658 4 {'result': 'tie', 'eliminator': 'Pokhara Avengers'}


In [27]:
match = load_match(NPL_DIR / "1462658.json")
for inn in match["innings"]:
    print(inn["team"], inn.get("super_over"), count_legal_balls(inn))


Biratnagar Kings None 120
Pokhara Avengers None 120
Pokhara Avengers True 6
Biratnagar Kings True 6


In [29]:
def innings_total(innings):
    total = 0
    for over in innings["overs"]:
        for delivery in over["deliveries"]:
            total += delivery["runs"]["total"]
    return total


In [30]:
print(innings_total(match["innings"][0]))
print(match["innings"][1]["target"])


139
{'overs': 20, 'runs': 140}


In [32]:
gate01_failed = []
gate01_skipped = []

for path in files:
    match = load_match(path)
    innings = match["innings"]

    if len(innings)<2:
        gate01_skipped.append((path.stem, "fewer than 2 innings"))
        continue

    target = innings[1].get("target")
    if target is None:
        gate01_skipped.append((path.stem, "no target"))
        continue

    expected = innings_total(innings[0]) + 1
    if target["runs"] != expected:
        gate01_failed.append((path.stem, target["runs"], expected))

print("checked:", len(files) - len(gate01_skipped))
print("skipped:", gate01_skipped)
print("failed:", gate01_failed)

checked: 64
skipped: []
failed: []


In [41]:
def innings_wickets(innings):
    total = 0
    for over in innings["overs"]:
        for delivery in over["deliveries"]:
            total += len(delivery.get("wickets",[]))
    return total

In [42]:
match = load_match(NPL_DIR / "1462658.json")
for inn in match["innings"]:
    print(inn["team"],innings_wickets(inn))

Biratnagar Kings 7
Pokhara Avengers 6
Pokhara Avengers 0
Biratnagar Kings 0


In [43]:
print(load_match(files[0])["info"]["outcome"])

{'winner': 'Janakpur Bolts', 'by': {'wickets': 8}}


In [44]:
print(load_match(NPL_DIR / "1462658.json")["info"]["outcome"])

{'result': 'tie', 'eliminator': 'Pokhara Avengers'}


In [45]:
gate02_failed = []
gate02_skipped = []

for path in files:
    match = load_match(path)
    innings = match["innings"]
    by = match["info"]["outcome"].get("by")

    if by is None:
        gate02_skipped.append((path.stem, "no margin"))
        continue

    if "wickets" in by:
        expected = 10 - innings_wickets(innings[1])
        if by["wickets"] != expected:
            gate02_failed.append((path.stem, "wickets", by["wickets"], expected))
    elif "runs" in by:
        expected = innings_total(innings[0]) - innings_total(innings[1])
        if by["runs"] != expected:
            gate02_failed.append((path.stem, "runs", by["runs"], expected))
    else:
        gate02_skipped.append((path.stem, "unknown margin"))

print("checked:", len(files) - len(gate02_skipped))
print("skipped:", gate02_skipped)
print("failed:", gate02_failed)


checked: 63
skipped: [('1462658', 'no margin')]
failed: []


In [47]:
from collections import Counter

kinds = Counter()
for path in files:
    match = load_match(path)
    for innings in match["innings"]:
        for over in innings["overs"]:
            for delivery in over["deliveries"]:
                for wicket in delivery.get("wickets", []):
                    kinds[wicket["kind"]] += 1

print(kinds.most_common())


[('caught', 487), ('bowled', 163), ('lbw', 114), ('run out', 77), ('stumped', 30), ('caught and bowled', 21), ('retired out', 2), ('hit wicket', 1), ('retired hurt', 1)]
